In [ ]:
import lightning as L
from pytorch_lightning.loggers import CSVLogger
from torchvision import transforms
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


### Improving over v9

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data"
DATA_PATH

In [ ]:
from src.model_training_pipeline import TripletSiameseModel  
from src.datamodule import TripletDataModule  
from src.random_seed_utils import seed_everything

In [ ]:
seed_everything()

### Changes

| # | Change                                                               | Expected Effect                                                                                                                                                                    |
| - | -------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1 | **Replaced top-k hard triplet mining with semi-hard triplet mining** | Selects triplets where `pos_dist < neg_dist < pos_dist + margin`, avoiding extremely hard or trivial samples and leading to more stable gradients and better embedding separation. |


In [ ]:
train_transforms = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.Grayscale(num_output_channels=3),

        transforms.RandomRotation(5),
        transforms.RandomAffine(
            degrees=0,
            translate=(0.02, 0.02),  
            scale=(0.95, 1.05),       
            shear=2                
        ),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),

        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])


val_transforms = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
])



In [ ]:
dm = TripletDataModule(
    data_dir=DATA_PATH,
    batch_size=64,
    num_workers=4,
    image_size=128,
    pin_memory=False,
    samples_per_epoch=10000,
    train_transformations=train_transforms,
    val_transformations=val_transforms,
    test_transformations=test_transforms
)

# Setup datasets (fit stage)
dm.setup(stage="fit")

In [ ]:
print(
    "Train:", len(dm.train_dataset),
    "Val:", len(dm.val_dataset),
    "Test:", len(getattr(dm, "test_dataset", []))
)

In [ ]:
len(dm.train_dataloader()), len(dm.val_dataloader())

In [ ]:
train_loader = dm.train_dataloader()
val_loader   = dm.val_dataloader()

print("Train batches per epoch:", len(train_loader))
print("Val batches per epoch:", len(val_loader))

In [ ]:
# from src.dataloader_utils import sanity_check_triplet_loader


# sanity_check_triplet_loader(loader=dm.train_dataloader(),split_name="train")
# sanity_check_triplet_loader(loader=dm.val_dataloader(),split_name="val")

In [ ]:
model = TripletSiameseModel(
    embedding_dim=256,
    lr=1e-4,
    margin=0.3
)

In [ ]:
logger = CSVLogger("train_logs", name="siamese_signature")

In [ ]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",      # metric to monitor
    patience=5,              # stop if no improvement after 5 epochs
    mode="min",              # we want to minimize val_loss
    verbose=True
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",      # metric to monitor
    dirpath="checkpoints",   # folder to save checkpoints
    filename="siamese-{epoch:02d}-{val_loss:.4f}",
    save_top_k=1,            # save only the best model
    mode="min",
    verbose=True
)

In [ ]:
trainer = L.Trainer(
    max_epochs=30,
    accelerator="auto",
    devices=1,
    logger=logger,
    log_every_n_steps=1,
    callbacks=[early_stop_callback, checkpoint_callback]
)

In [ ]:
trainer.fit(model, datamodule=dm)

In [ ]:
checkpoint_path = "train_logs/siamese_signature/version_10/checkpoints/epoch=22-step=3588.ckpt"
model = TripletSiameseModel.load_from_checkpoint(embedding_dim=256,checkpoint_path=checkpoint_path)
model.eval()

In [ ]:
from src.train_metrics_utils import plot_triplet_training_metrics


metrics = plot_triplet_training_metrics(csv_path="train_logs/siamese_signature/version_10/metrics.csv")


In [ ]:
dm.setup(stage="test") 

In [ ]:
test_loader = dm.test_dataloader()
len(dm.test_dataset)

In [ ]:
len(test_loader)

In [ ]:
trainer.test(model, datamodule=dm)



In [ ]:
from src.inference_utils import plot_triplet_distance_distributions


plot_triplet_distance_distributions(
    model.test_pos_distances,
    model.test_neg_distances
)

In [ ]:
from src.inference_utils import find_best_threshold


t, acc = find_best_threshold(
    model.test_pos_distances,
    model.test_neg_distances
)

print("Best threshold:", t)
print("Best accuracy:", acc)

In [ ]:
from src.inference_utils import evaluate_with_best_threshold


cm = evaluate_with_best_threshold(
    model.test_pos_distances,
    model.test_neg_distances
)